# LS-NEQ 07_lambda_dynamics — λ-Dynamik im Aktivierungsraum

**Dossier-Stufe H** (`lsne.experiments.STAGES["H"]`, siehe `docs/DESIGN.md` §8 und `docs/STAGES.md`).
Dieses Notebook wird von `scripts/build_notebooks.py` erzeugt; Änderungen bitte dort
vornehmen und neu generieren (`python scripts/build_notebooks.py`).

**Zweck.** Biggio et al. (2013/2017), Algorithmus 1 im Aktivierungsraum (`docs/DESIGN.md` §4.6): von der Dossier-Stelle (Block-Eingang des letzten Blocks, letzte Position) eines K=0-Starts aus wird F(h) = ĝ(h) − λ·p̂(h) per Einheitsschritten minimiert — ĝ eine Diskriminante (`target_logit`: −log p(Ziel); `probe`: lineare Sprach-Probe; `entropy`; `sae_feature`), p̂ eine KDE über echte K=8/C1-Zustände (Mahalanobis-gewhitet). λ = 0 überquert die Grenze schnellstmöglich, aber ohne Datenstützung; λ > 0 erzwingt Mimikry. Pro λ werden Pfadgeometrie, Datenstützung des Endpunkts (Mahalanobis zur Zielwolke, k-NN, SAE-FVU), Wirkung (B, Entropie, p_freq) und die SAE-Features gemessen; Spearman-ρ mit λ über den Sweep, Familien-Bootstrap; zusätzlich B(x)(λ) für State 2143.

**Eingaben.** Stimulusbank, Richtungsbank (für `probe`: `lang_probe`), K=8/C1-Realisierungswolke (Stufe G aus `ctx`, sonst neu gebaut), optional SAE (`USE_SAE=True` auf step143000; Rauchtest: zufälliger Platzhalter-SAE, markiert `sae_source = synthetic_random`), Anker für State 2143.

**Ausgaben.** Tabellen `lambda_paths`, `lambda_steps`, `lambda_corr`, `lambda_boot`, `lambda_features`, `lambda_features_summary`, `lambda_original`.
Alle Tabellen liegen als CSV (+ Parquet) im Laufverzeichnis
`<RUNS_ROOT>/H/<modell>@<revision>/` zusammen mit `manifest.json`
(Git-SHA, Modell, dtype, Hook, Anker-Hashes, Zeitstempel); Abbildungen unter
`<OUT_ROOT>/figures/07_lambda_dynamics/`; am Ende eine kurze `summary_07_lambda_dynamics_*.json` in `OUT_ROOT`.

**Ablauf.** Setup → Drive → Konfiguration → Stufe-H-Parameter → Modell → Stufe H → Pfade (g, Dichte, F) → Endpunkt-Größen vs. λ → Spearman mit Bootstrap → SAE-Feature-Kandidaten → State 2143 → Zusammenfassung.

**Präregistrierung.** Kontrast 10 (`lambda_boot`-Zeilen mit `preregistered = True`: Diskriminante `target_logit`, ρ(λ, SAE-FVU) < 0, ρ(λ, Schritte bis zur Grenze) offen, ρ(λ, Mahalanobis zur Zielwolke) < 0; Feature-Kandidaten |ρ| ≥ 0,7 in `lambda_features`) und Kontrast 11 (`lambda_original`: Vorzeichen von B bei λ_rel = 3 gegenüber λ = 0, `same_sign`). Kandidaten werden auf Entwicklungsfamilien eingefroren und auf Testfamilien geprüft.

Hinweise: Die Zellen sind gegen Fehler abgesichert (ein fehlgeschlagener Schritt druckt
den Traceback und das Notebook läuft weiter). Mit `SMOKE = True` läuft alles in
< 3 min auf CPU mit `pythia-14m` (Rauchtest, keine wissenschaftliche Aussage).
Mit `REUSE = True` werden vorhandene Ergebnisse (manifest.json mit `status: ok`)
geladen statt neu gerechnet — so lässt sich nach einem Colab-Abbruch fortsetzen.


In [ ]:
# --- Setup: Repository holen / aktualisieren und installieren -------------------------
# Runs inside Colab (clone into /content/lsne-repo) and inside an existing checkout
# (local Jupyter, tests): a checkout is detected by lsne/__init__.py + pyproject.toml.
import importlib.util
import os
import pathlib
import subprocess
import sys

GIT_URL = os.environ.get("LSNE_GIT_URL", "https://github.com/Erikiss/Language-Switching-.git")
GIT_BRANCH = os.environ.get("LSNE_GIT_BRANCH", "claude/language-switching-non-equilibrium-ejpz0q")
REPO_DIR = os.environ.get("LSNE_REPO_DIR", "/content/lsne-repo")


def _sh(cmd: str) -> int:
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode


_here = pathlib.Path.cwd().resolve()
_local = next((p for p in [_here, *_here.parents]
               if (p / "lsne" / "__init__.py").exists() and (p / "pyproject.toml").exists()), None)
if _local is not None:
    REPO_DIR = str(_local)
    print("Vorhandener Checkout:", REPO_DIR)
elif pathlib.Path(REPO_DIR, ".git").exists():
    _sh(f"git -C {REPO_DIR} fetch -q --depth 1 origin {GIT_BRANCH} && "
        f"git -C {REPO_DIR} checkout -q {GIT_BRANCH} && git -C {REPO_DIR} pull -q --ff-only origin {GIT_BRANCH}")
else:
    _sh(f"git clone -q --depth 1 --branch {GIT_BRANCH} {GIT_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if _local is None or importlib.util.find_spec("lsne") is None:
    _sh(f"{sys.executable} -m pip install -q -r requirements.txt")
    _sh(f"{sys.executable} -m pip install -q -e .")
import lsne
print("lsne", lsne.__version__, "aus", REPO_DIR, "| Branch", GIT_BRANCH)
print("Git-SHA:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())


In [ ]:
# --- Optional: TNCO (Google Research) ---------------------------------------------------
# TNCO optimises the contraction ORDER of the linearised lattice network (stage E).
# It needs the Boost headers and a C++ build (a few minutes on Colab).  Without TNCO
# lsne.tn_lattice.optimize_path falls back to opt_einsum ("dp" for <= 12 tensors,
# else "greedy"); the contracted response is identical, only the path cost / runtime
# columns of tn_paths differ ("fallback_reason" says why).  Stages A-D never need it.
INSTALL_TNCO = False   # True: bauen (nur für Notebook 04 / Stufe E sinnvoll)

try:
    import tnco
    print("tnco", getattr(tnco, "__version__", "?"), "ist bereits installiert")
except ImportError:
    if INSTALL_TNCO:
        _sh("apt-get install -y -q libboost-dev > /tmp/apt_boost.log 2>&1 || cat /tmp/apt_boost.log | tail -5")
        _sh(f'{sys.executable} -m pip install -q "tnco @ git+https://github.com/google-research/tnco"')
        try:
            import tnco
            print("tnco", getattr(tnco, "__version__", "?"), "installiert")
        except ImportError as e:
            print("TNCO konnte nicht gebaut werden:", repr(e), "-> Fallback opt_einsum (siehe docs/RUNBOOK_COLAB.md)")
    else:
        print("TNCO nicht installiert -> lsne.tn_lattice nutzt opt_einsum (dp/greedy) als Fallback.")


In [ ]:
# --- Google Drive einbinden (außerhalb von Colab wird lokal geschrieben) ---------------
# On Colab the results MUST go to Drive: a refused or failed mount stops the notebook
# here (the local disk of the runtime is deleted with the session).  Outside Colab, or
# with LSNE_ALLOW_LOCAL=1, the outputs go to $LSNE_RUNS or ./runs.
import os
import pathlib

try:
    from google.colab import drive            # type: ignore
except ImportError:                           # not on Colab
    drive = None
    print("Kein Colab -> lokale Ausgabe")
if drive is not None and os.environ.get("LSNE_ALLOW_LOCAL") != "1":
    drive.mount("/content/drive")             # confirm the permission dialog in this tab
    if not pathlib.Path("/content/drive/MyDrive").is_dir():
        raise RuntimeError("Google Drive ist nicht eingebunden: die Ergebnisse würden nur auf der "
                           "Laufzeit liegen und mit ihr gelöscht. Diese Zelle erneut ausführen und den "
                           "Drive-Zugriff im Dialog erlauben.")

from lsne.io import drive_root

OUT_ROOT = drive_root()                       # Colab: /content/drive/MyDrive/LSNE, sonst $LSNE_RUNS oder ./runs
RUNS_ROOT = os.path.join(OUT_ROOT, "runs")    # Laufverzeichnisse wie in jobs/shards.yaml (--out)
os.makedirs(RUNS_ROOT, exist_ok=True)
if OUT_ROOT.startswith("/content/drive"):
    # share the anchor / unigram caches between sessions (see jobs/shards.yaml defaults.env)
    os.environ.setdefault("LSNE_CACHE", os.path.join(OUT_ROOT, "cache"))
print("OUT_ROOT  =", OUT_ROOT)
print("RUNS_ROOT =", RUNS_ROOT)
print("LSNE_CACHE =", os.environ.get("LSNE_CACHE", "./cache"))


In [ ]:
# --- Konfiguration (hier anpassen) ------------------------------------------------------
NOTEBOOK = "07_lambda_dynamics"
STAGE = "H"
MODEL = "EleutherAI/pythia-160m"   # HF-Modellname (State 2143: pythia-160m)
REVISION = "step98000"             # HF-Revision: step98000 / step99000 (Eingriff), step143000 (SAE)
DTYPE = "float32"                  # "float32" (Standard) | "bfloat16" (eigener Lauf, eigene Null-Dosis-Baseline)
DEVICE = "auto"                    # "auto" | "cuda" | "cpu"
FAMILIES = None                    # z. B. ["F01", "F02"] oder ["DEV01A"]; None = alle Familien (F01-F12, DEV01-DEV04)
SMOKE = False                      # True: Mini-Lauf mit pythia-14m, 1 Familie, 3 Varianten (< 3 min CPU)
SEED = 0
REUSE = True                       # vorhandene Ergebnisse (manifest.json, status ok) laden statt neu rechnen
USE_SAE = False                    # SAE-Feature-Richtungen (EleutherAI/sae-pythia-160m-32k; nur bei step143000 zertifizierbar)

from lsne.config import ModelSpec, RunConfig
from lsne.experiments import common as C

if SMOKE:
    MODEL, REVISION, DTYPE = "EleutherAI/pythia-14m", "step98000", "float32"
    FAMILIES = FAMILIES or ["DEV01"]
spec = ModelSpec(name=MODEL, revision=REVISION, dtype=DTYPE, device=DEVICE)
cfg = RunConfig(model=spec, families=FAMILIES, seed=SEED, out_root=RUNS_ROOT)
limits = C.Limits()
if SMOKE:
    cfg, limits = C.smoke_config(cfg), C.Limits.smoke()
limits.use_sae = bool(USE_SAE) and not SMOKE
limits.progress = True

_fam = f" --families {','.join(FAMILIES)}" if FAMILIES else ""
CLI_COMMAND = (f"python -m lsne.cli run --stage {STAGE} --model {MODEL} --revision {REVISION} --dtype {DTYPE}"
               f" --device {DEVICE} --out {RUNS_ROOT} --seed {SEED}{_fam}{' --smoke' if SMOKE else ''}"
               f"{' --use-sae' if limits.use_sae else ''}")
print("Modell:", spec.tag(), "| Familien:", FAMILIES or "alle", "| SMOKE:", SMOKE)
print("CLI-Äquivalent:\n  " + CLI_COMMAND)


In [ ]:
# --- Stufe-H-Parameter (Budget ≈ Starts × Diskriminanten × λ-Stufen × Schritte Forward+Backward) --
H_N_STEPS = None       # Schritte pro Pfad (Standard limits.h_n_steps = 100; kleine GPU: 30)
H_MAX_STARTS = None    # Startzustände (K=0-Formulierungen) pro Familie (Standard 4)
if H_N_STEPS is not None:
    limits.h_n_steps = int(H_N_STEPS)
if H_MAX_STARTS is not None:
    limits.h_max_starts = int(H_MAX_STARTS)
print(f"λ_rel-Sweep {limits.h_lambdas_rel}; Pfad: {limits.h_n_steps} Schritte × {limits.h_step_rel}·‖h₀‖ in der "
      f"ℓ₂-Kugel {limits.h_dmax_rel}·‖h₀‖; Starts/Familie {limits.h_max_starts}; Diskriminanten {limits.h_discriminants}; "
      f"KDE k_nearest={limits.h_knn}, Rang {limits.h_cov_rank}; Grenzabstand {limits.h_cross_nats} nats; "
      f"SAE-Platzhalter bei Bedarf: {limits.synthetic_sae_if_missing}")


In [ ]:
# --- Modell laden -------------------------------------------------------------------
import torch
from lsne.model import load_model

mb = load_model(spec)
print(f"{spec.tag()}: {mb.n_layers} Blöcke, d_model={mb.d_model}, {mb.n_heads} Heads, "
      f"Vokabular={mb.vocab}, device={mb.device}, dtype={mb.dtype}")
print("Dossier-Eingriffsstelle (verallgemeinert):", C.dossier_site(mb).key,
      "| Tiefen-Injektion (D1b / Stufe E):", C.depth_site(mb).key)


In [ ]:
# --- Runner: Stufen direkt über run() ausführen (Ergebnisse bleiben im Speicher) --------
import contextlib
import json
import time
import traceback

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from lsne.experiments import STAGES, STAGE_TITLES
from lsne.io import RunDir, load_df, read_manifest

ctx: dict = {}        # shared objects between stages (variants, original, direction bank)
results: dict = {}    # stage -> {table name: DataFrame}
report: dict = {}     # stage -> status / runtime / error
FIG_DIR = os.path.join(OUT_ROOT, "figures", NOTEBOOK)
os.makedirs(FIG_DIR, exist_ok=True)


def run_stage(stage: str, reuse=None, **kw):
    """Run one stage; on failure print the traceback, write error.txt and continue."""
    reuse = REUSE if reuse is None else reuse
    rd = RunDir(cfg.out_root, stage, cfg.model)
    t0 = time.perf_counter()
    if reuse and rd.manifest_path.exists():
        try:
            m = read_manifest(rd)
            if m.get("status") == "ok" and m.get("tables"):
                results[stage] = {name: load_df(rd, name) for name in m["tables"]}
                report[stage] = dict(status="reused", run_dir=str(rd.path), seconds=0.0, error="")
                print(f"Stufe {stage}: vorhandene Ergebnisse geladen aus {rd.path} "
                      f"(REUSE=False erzwingt Neuberechnung)")
                return results[stage]
        except Exception as e:  # unreadable previous run: recompute
            print("Wiederverwendung nicht möglich:", repr(e))
    print(f"Stufe {stage} ({STAGE_TITLES[stage]}) -> {rd.path}", flush=True)
    try:
        results[stage] = STAGES[stage].run(cfg, mb, rd, ctx=ctx, limits=limits, **kw)
        report[stage] = dict(status="ok", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error="")
        print(f"Stufe {stage} ok in {report[stage]['seconds']:.0f} s: {', '.join(sorted(results[stage]))}")
    except Exception as e:
        traceback.print_exc()
        C.write_error(rd, stage, e, cfg, mb)
        report[stage] = dict(status="error", run_dir=str(rd.path), seconds=time.perf_counter() - t0, error=repr(e))
        results[stage] = {}
        print(f"Stufe {stage} FEHLGESCHLAGEN: {e!r} (error.txt in {rd.path})")
    return results[stage]


def table(stage: str, name: str) -> pd.DataFrame:
    """Table of a stage result or an empty DataFrame (keeps the plot cells robust)."""
    df = results.get(stage, {}).get(name)
    return df if isinstance(df, pd.DataFrame) else pd.DataFrame()


def show(df, n: int = 40):
    """display() in Jupyter / Colab, print() elsewhere."""
    if isinstance(df, pd.DataFrame):
        df = df.head(n)
    try:
        display(df)          # noqa: F821 - provided by IPython
    except NameError:
        print(df.to_string() if hasattr(df, "to_string") else df)


def savefig(name: str):
    path = os.path.join(FIG_DIR, f"{name}_{spec.tag().replace(':', '_').replace('@', '_')}.png")
    plt.gcf().savefig(path, dpi=120, bbox_inches="tight")
    print("Abbildung:", path)


@contextlib.contextmanager
def guarded(title: str):
    """Print the traceback of a failing cell section and carry on."""
    print("==", title)
    try:
        yield
    except Exception:
        traceback.print_exc()
        print(f"[{title}] übersprungen (siehe Traceback)")


In [ ]:
# --- Substufen-Bericht: die Stufen G-L fangen Fehler pro Substufe ab (docs/STAGES.md) ----
# A failed substage is recorded in manifest.json -> substages.errors (traceback in
# substage_errors.txt), its table stays empty and the stage goes on; the manifest's
# `prereg` map names the table / rows that carry every pre-registered contrast.
def substage_report(stage: str) -> dict:
    """Print the substage report and the pre-registration map of a G-L manifest."""
    rd = RunDir(cfg.out_root, stage, cfg.model)
    if not rd.manifest_path.exists():
        print(f"Stufe {stage}: kein manifest.json in {rd.path}")
        return {}
    m = read_manifest(rd)
    sub = m.get("substages") or {}
    rt = m.get("runtime_s")
    print(f"Stufe {stage}: Status {m.get('status')}" + (f", {rt:.0f} s" if isinstance(rt, (int, float)) else "")
          + f", Substufen fehlgeschlagen: {sub.get('n_failed', '?')}"
          + (" | STUB: lsne.transplant nicht importierbar" if m.get("stub") else ""))
    for name, err in (sub.get("errors") or {}).items():
        print(f"  FEHLER in Substufe {name}: {err}")
    if sub.get("n_failed"):
        print(f"  Tracebacks: {rd.path / 'substage_errors.txt'}")
    for k, v in (m.get("prereg") or {}).items():
        print(f"  Präregistrierung {k}: {v}")
    return m


def truthy(s: pd.Series) -> pd.Series:
    """Boolean view of a flag column (bool in memory, 'True'/'False' strings after a csv reload)."""
    return s.astype(str).str.lower().eq("true")


In [ ]:
# --- Stufe H ausführen ----------------------------------------------------------------
# CLI-Äquivalent:
#   python -m lsne.cli run --stage H --model EleutherAI/pythia-160m --revision step98000 --out <RUNS_ROOT> [--use-sae]
# (--stage G,H shares the realization clouds; with --use-sae on step143000 the sae_feature
# discriminant and the feature correlations use the published SAE instead of a placeholder)
res_H = run_stage("H")
print({k: len(v) for k, v in res_H.items()})
manifest_H = substage_report("H")
if manifest_H:
    print("Starts:", manifest_H.get("n_starts"), "| Zielzustände:", manifest_H.get("n_target"), "| KDE:", manifest_H.get("kde"),
          "| SAE:", manifest_H.get("sae_source"))


In [ ]:
# --- Pfade im Aktivierungsraum: g, Dichte und F = g − λ·Dichte pro Schritt (erster Start) -------
with guarded("λ-Pfade"):
    steps = table("H", "lambda_steps")
    tag0 = str(steps["tag"].iloc[0])
    discs = list(dict.fromkeys(steps["discriminant"]))
    fig, axes = plt.subplots(len(discs), 3, figsize=(15, 3.2 * len(discs)), squeeze=False)
    for i, disc in enumerate(discs):
        g = steps[(steps["tag"] == tag0) & (steps["discriminant"] == disc)]
        for lam_rel, p in sorted(g.groupby("lam_rel")):
            p = p.sort_values("step")
            for ax, col in zip(axes[i], ["g", "density", "F"]):
                ax.plot(p["step"], p[col], marker=".", ms=3, label=f"λ_rel={lam_rel:g}")
        axes[i, 0].set_ylabel(disc)
        axes[i, 0].set_title("Diskriminante g (g < 0: Grenze überquert)", fontsize=9)
        axes[i, 1].set_title("Zielklassen-Dichte p̂ (KDE, Mahalanobis)", fontsize=9)
        axes[i, 2].set_title("F = g − λ·p̂ (Alg. 1 minimiert F)", fontsize=9)
        axes[i, 0].axhline(0, color="k", lw=0.6)
    for ax in axes[-1]:
        ax.set_xlabel("Schritt")
    axes[0, 2].legend(fontsize=7)
    fig.suptitle(f"Start {tag0}: λ = 0 steigt schnellstmöglich ab, λ > 0 bleibt in der Datenstützung (Mimikry)", fontsize=10, y=1.01)
    savefig("lambda_paths")
    plt.show()


In [ ]:
# --- Endpunkt-Größen als Funktion von λ: Schritte bis zur Grenze, Mahalanobis, SAE-FVU, B -------
with guarded("Endpunkte vs. λ"):
    paths = table("H", "lambda_paths")
    quants = [q for q in ["steps_to_cross", "mahal_to_ref_mean", "knn_mean_dist", "sae_fvu", "B", "path_length"] if q in paths.columns]
    fig, axes = plt.subplots(1, len(quants), figsize=(3.4 * len(quants), 3.6), squeeze=False)
    for ax, q in zip(axes[0], quants):
        for disc, g in paths.groupby("discriminant"):
            m = g.groupby("lam_rel")[q].agg(["mean", "std"])
            ax.errorbar(m.index, m["mean"], yerr=m["std"].fillna(0), marker="o", ms=4, capsize=2, label=disc)
        ax.set_xscale("symlog", linthresh=0.1)
        ax.set_xlabel("λ_rel")
        ax.set_title(q, fontsize=9)
    axes[0, 0].legend(fontsize=7)
    fig.suptitle("Mittel ± SD über Starts (steps_to_cross: NaN = Grenze im Budget nicht erreicht)", fontsize=10)
    savefig("lambda_endpoints")
    plt.show()
    print("Anteil überquerter Grenzen pro (Diskriminante, λ_rel):")
    show(paths.assign(crossed=truthy(paths["crossed"]).astype(float)).pivot_table(index="discriminant", columns="lam_rel", values="crossed", aggfunc="mean"))


In [ ]:
# --- Spearman-ρ(λ, ·) pro Diskriminante mit Familien-Bootstrap (Kontrast 10) -----------------
with guarded("Spearman-Korrelationen"):
    lb = table("H", "lambda_boot")
    pre = lb[truthy(lb["preregistered"])]
    print("Präregistrierte Zeilen (target_logit; erwartete Vorzeichen: sae_fvu < 0, mahal_to_ref_mean < 0, steps_to_cross offen):")
    show(pre[["discriminant", "quantity", "expected_sign", "sign_as_expected", "mean", "ci_low", "ci_high", "n_clusters"]])
    quants = [q for q in ["steps_to_cross", "mahal_to_ref_mean", "knn_mean_dist", "sae_fvu", "local_log_volume", "B", "entropy",
                          "kl_from_base", "path_length", "tortuosity"] if q in set(lb["quantity"])]
    sub = lb[lb["quantity"].isin(quants)]
    discs = list(dict.fromkeys(sub["discriminant"]))
    fig, ax = plt.subplots(figsize=(1.1 * len(quants) + 3, 4))
    x = np.arange(len(quants))
    w = 0.8 / max(1, len(discs))
    for i, disc in enumerate(discs):
        g = sub[sub["discriminant"] == disc].set_index("quantity").reindex(quants)
        ax.bar(x + (i - (len(discs) - 1) / 2) * w, g["mean"], w,
               yerr=[(g["mean"] - g["ci_low"]).clip(lower=0).fillna(0), (g["ci_high"] - g["mean"]).clip(lower=0).fillna(0)],
               capsize=2, label=disc)
    ax.set_xticks(x)
    ax.set_xticklabels(quants, rotation=40, ha="right", fontsize=8)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_ylabel("Spearman ρ(λ, Größe), Familienmittel")
    ax.set_title("Welche Pfadgrößen korrelieren monoton mit λ?", fontsize=10)
    ax.legend(fontsize=7)
    savefig("lambda_spearman")
    plt.show()


In [ ]:
# --- SAE-Features vs. λ: Kandidaten |ρ| ≥ 0,7; nur bei λ=0 bzw. nur bei λ>0 aktive Features ----
with guarded("SAE-Feature-Korrelationen"):
    fs = table("H", "lambda_features_summary")
    paths = table("H", "lambda_paths")
    if fs.empty:
        print("Keine SAE-Feature-Tabelle (kein SAE: USE_SAE=True auf step143000; im Rauchtest ein Platzhalter-SAE).")
    else:
        src = str(paths["sae_source"].iloc[0]) if len(paths) else "?"
        print("SAE-Quelle:", src, "(synthetic_random = Platzhalter: Funktionstest, kein Befund)")
        cand = fs[truthy(fs["candidate_all"])]
        print(f"{len(cand)} Kandidaten-Features (|ρ| ≥ 0,7 bei allen Starts):")
        show(cand.sort_values("rho_mean"), n=40)
        fig, ax = plt.subplots(figsize=(7, 3.6))
        for disc, g in fs.groupby("discriminant"):
            ax.scatter(g["feature"], g["rho_mean"], s=12, label=disc, alpha=0.7)
        ax.axhline(0.7, color="r", lw=0.7, ls="--")
        ax.axhline(-0.7, color="r", lw=0.7, ls="--")
        ax.axhline(0, color="k", lw=0.5)
        ax.set_xlabel("SAE-Feature (Latent-Index)")
        ax.set_ylabel("mittleres ρ(λ, Aktivierung am Endpunkt)")
        ax.set_title("Feature-Aktivierung vs. λ (gestrichelt: Kandidatenschwelle |ρ| = 0,7)", fontsize=9)
        ax.legend(fontsize=7)
        savefig("lambda_features")
        plt.show()
        print("Nur bei λ=0 aktiv (Artefakt-Kandidaten) bzw. nur bei λ>0 aktiv (Repräsentations-Kandidaten):")
        show(fs[(fs["n_active_only_lambda0"] > 0) | (fs["n_active_only_lambda_pos"] > 0)]
             .sort_values(["n_active_only_lambda_pos", "n_active_only_lambda0"], ascending=False), n=30)


In [ ]:
# --- Kontrast 11: B(x)(λ) für State 2143 — bleibt der 98k/99k-Nutzen auf der Datenmannigfaltigkeit bestehen? --
with guarded("State 2143: B(λ)"):
    orig = table("H", "lambda_original")
    if orig.empty:
        print("Keine Original-Tabelle (Anker nicht verfügbar oder zu wenige Zielzustände).")
    else:
        r = orig.iloc[0]
        print(f"B(λ=0) = {r['B_lambda0']:+.4f} nats, B(λ_rel=3) = {r['B_lambda3']:+.4f} nats, gleiches Vorzeichen: {r['same_sign']}; "
              f"Zielwolke: {r['target_cloud']} (n_orig={r['n_target_orig']}, n={r['n_target']})")
        g = orig.sort_values("lam_rel")
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
        a1.plot(g["lam_rel"], g["B"], "o-", label="B = ln p_δ(bey) − ln p₀(bey)")
        a1.axhline(0, color="k", lw=0.6)
        a1.set_xscale("symlog", linthresh=0.1)
        a1.set_xlabel("λ_rel")
        a1.set_ylabel("B [nats]")
        a1.set_title("State 2143 (Ziel 22905 „bey“), Diskriminante target_logit", fontsize=9)
        a1.legend(fontsize=7)
        a2.plot(g["lam_rel"], g["mahal_to_ref_mean"], "s-", color="C1", label="Mahalanobis zur Zielwolke")
        a2.plot(g["lam_rel"], g["sae_fvu"], "^-", color="C2", label="SAE-FVU am Endpunkt")
        a2.set_xscale("symlog", linthresh=0.1)
        a2.set_xlabel("λ_rel")
        a2.set_title("Datenstützung des Endpunkts", fontsize=9)
        a2.legend(fontsize=7)
        savefig("lambda_original_2143")
        plt.show()
        show(g[["lam_rel", "lam", "n_steps", "steps_to_cross", "crossed", "B", "entropy", "p_freq", "mahal_to_ref_mean", "sae_fvu"]])


In [ ]:
# --- Kurze Laufzusammenfassung nach OUT_ROOT schreiben ---------------------------------
with guarded("Zusammenfassung schreiben"):
    import datetime

    summary = dict(
        notebook=NOTEBOOK, stage=STAGE, model=spec.name, revision=spec.revision, dtype=spec.dtype,
        device=str(mb.device), n_layers=mb.n_layers, d_model=mb.d_model, smoke=bool(SMOKE),
        families=FAMILIES, seed=SEED, out_root=OUT_ROOT, runs_root=RUNS_ROOT, cli_command=CLI_COMMAND,
        timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
        report=report,
        tables={st: {n: int(len(df)) for n, df in r.items()} for st, r in results.items()},
        figures=sorted(f for f in os.listdir(FIG_DIR) if f.endswith(".png")),
    )
    path = os.path.join(OUT_ROOT, f"summary_{NOTEBOOK}_{spec.tag().replace(':', '_').replace('@', '_')}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(C.to_jsonable(summary), f, indent=1, ensure_ascii=False)
    print("Zusammenfassung:", path)
    print(json.dumps(C.to_jsonable(summary), indent=1, ensure_ascii=False)[:3000])
